In [11]:
import os
import json
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_tavily import TavilySearch
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools.arxiv.tool import ArxivQueryRun
from mlx_lm import load, generate

In [12]:
# ---------------------------
# Load MLX Model (GPU via Metal)
# ---------------------------
print("Loading MLX model...")
model, tokenizer = load("mlx-community/Qwen2.5-3B-Instruct-4bit")
print("Model loaded successfully!")

Loading MLX model...


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 78316.88it/s]



Model loaded successfully!


In [13]:
# ---------------------------
# API Key (Better: use .env file)
# ---------------------------
if "TAVILY_API_KEY" not in os.environ:
    os.environ["TAVILY_API_KEY"] = "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"

In [14]:
# ---------------------------
# Agent State
# ---------------------------
class AgentState(TypedDict):
    research_topic: str
    retrieved_docs: List[str]
    knowledge_context: dict

In [20]:
# ---------------------------
# Reader Prompt
# ---------------------------
import re

READER_SYSTEM_PROMPT = (
    "You are a research analysis assistant. "
    "You MUST respond with ONLY a single valid JSON object and nothing else. "
    "No markdown, no explanation, no commentary — just the JSON object."
)

READER_USER_TEMPLATE = """Analyze these research snippets and return a JSON object with exactly these keys:

- "core_domain": a short string describing the research domain
- "key_findings": a list of 2-4 key finding strings
- "methodologies_used": a list of methodologies/tools mentioned
- "identified_gaps": a list of objects, each with "type" (one of "limit_of_scope", "conflicting_evidence", "explicit_future_work") and "description"
- "novelty_score": a float between 0.0 and 1.0

Research Snippets:
{docs_text}

Respond with ONLY the JSON object:"""


def extract_json(text: str) -> dict:
    """Robustly extract a JSON object from model output that may contain
    markdown fences, preamble text, or trailing commentary."""

    # 1. Try the raw text first
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # 2. Strip markdown code fences (```json ... ``` or ``` ... ```)
    fence_pattern = r"```(?:json)?\s*\n?(.*?)```"
    match = re.search(fence_pattern, text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1).strip())
        except json.JSONDecodeError:
            pass

    # 3. Find the first { ... } block (greedy)
    brace_match = re.search(r"\{.*\}", text, re.DOTALL)
    if brace_match:
        try:
            return json.loads(brace_match.group(0))
        except json.JSONDecodeError:
            pass

    # 4. Give up — return error with raw output
    return {"error": "Could not extract valid JSON", "raw_output": text}

In [16]:
# ---------------------------
# Search Node
# ---------------------------
def search_node(state: AgentState):
    topic = state["research_topic"]
    print(f"\n[Search Agent] Searching for: {topic}")

    docs = []

    try:
        arxiv = ArxivQueryRun(
            api_wrapper=ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=1000)
        )
        docs.append("ArXiv Source:\n" + arxiv.run(topic))
    except Exception as e:
        print("ArXiv error:", e)

    try:
        tavily = TavilySearch(max_results=2)
        results = tavily.invoke({"query": topic})
        for r in results.get("results", []):
            docs.append(f"Web Source ({r.get('url', 'unknown')}):\n{r.get('content', '')}")
    except Exception as e:
        print("Tavily error:", e)

    return {"retrieved_docs": docs}


In [21]:
# ---------------------------
# Reader Node (MLX version)
# ---------------------------
def reader_node(state: AgentState):
    print("\n[Reader Agent] Analyzing with MLX...")

    docs_text = "\n\n".join(state["retrieved_docs"])

    # Build a proper chat-template prompt for the instruct model
    messages = [
        {"role": "system", "content": READER_SYSTEM_PROMPT},
        {"role": "user", "content": READER_USER_TEMPLATE.format(docs_text=docs_text)},
    ]
    chat_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    # Generate (default sampler = greedy / argmax)
    response = generate(
        model,
        tokenizer,
        prompt=chat_prompt,
        max_tokens=1024,
    )

    print(f"[Reader Agent] Raw model output ({len(response)} chars):")
    print(response[:500])

    parsed = extract_json(response)
    return {"knowledge_context": parsed}

In [22]:
# ---------------------------
# Build Graph
# ---------------------------
def build_graph():
    workflow = StateGraph(AgentState)
    workflow.add_node("search_agent", search_node)
    workflow.add_node("reader_agent", reader_node)
    workflow.set_entry_point("search_agent")
    workflow.add_edge("search_agent", "reader_agent")
    workflow.add_edge("reader_agent", END)
    return workflow.compile()

In [19]:
# ---------------------------
# Main Execution
# ---------------------------
if __name__ == "__main__":
    app = build_graph()

    print("\n--- Autonomous Research Scientist (MLX Version) ---")
    topic = input("Enter research topic: ")

    if topic:
        inputs = {
            "research_topic": topic,
            "retrieved_docs": [],
            "knowledge_context": {}
        }

        result = app.invoke(inputs)

        print("\n" + "=" * 50)
        print("FINAL KNOWLEDGE CONTEXT")
        print("=" * 50)
        print(json.dumps(result["knowledge_context"], indent=2))
        print("=" * 50)



--- Autonomous Research Scientist (MLX Version) ---

[Search Agent] Searching for: Transformer

[Search Agent] Searching for: Transformer

[Reader Agent] Analyzing with MLX...

[Reader Agent] Analyzing with MLX...

FINAL KNOWLEDGE CONTEXT
{
  "error": "Model did not return valid JSON",
  "raw_output": "Web Source (https://www.electricaltechnology.org/transformer-electronics/):\nTransformers are electrical devices that transfer electrical energy from one circuit to another through electromagnetic induction. They are essential components in electrical power systems, allowing the efficient transfer of electrical energy over long distances. The primary winding of a transformer is connected to the power source, while the secondary winding is connected to the load. The transformer's core, made of ferromagnetic material, creates a magnetic field that induces a voltage in the secondary winding. The turns ratio between the primary and secondary windings determines the voltage transformation. T